In [1]:
import os
import sys
sys.path.insert(0, '../..')
import numpy as np
import matplotlib.pyplot as plt
import mssfp
import deepssfp
import deepssfp.analysis

In [ ]:
modes=['BandRemoval:4', 'BandRemoval:2', 'SyntheticBanding', 'SuperFOV', 'SuperFOVi']
model_dir = "D:/DeepSSFP/"
model_name="knee_metrics"

In [ ]:
path = 'D:/MRI/2024_Krithika_KneeStudy'
filters = [
    'HV1_lk_r1', 'HV2_lk_r1', 'HV3_lk_r1', 'HV4_lk_r1', 'HV5_lk_r1', 'HV6_lk_r1', 'HV7_lk_r1', 'HV8_lk_r1',
    'HV1_rk_r1', 'HV2_rk_r1', 'HV3_rk_r1', 'HV4_rk_r1', 'HV5_rk_r1', 'HV6_rk_r1', 'HV7_rk_r1', 'HV8_rk_r1',]

dataset = deepssfp.dataloader.read_complex_dicom_datasets(path, filters = filters)
print(dataset['M'.shape])

100%|██████████| 4/4 [00:10<00:00,  2.53s/it]


Dataset loaded: (1280, 416, 416, 4)
Memory size: 7.08837376 GB


In [10]:
print(dataset['M'].shape, dataset['M'].dtype)

(1280, 416, 416, 4) complex64


In [11]:
import numpy as np
from skimage.transform import resize      # pip install scikit-image

def resize_complex(img, out_hw=(256, 256), order=3):
    """Resize a single complex 2-D image."""
    real = resize(img.real, out_hw, order=order,
                  mode='reflect', anti_aliasing=True, preserve_range=True)
    imag = resize(img.imag, out_hw, order=order,
                  mode='reflect', anti_aliasing=True, preserve_range=True)
    return (real + 1j*imag).astype(np.complex64)

def batch_resize(data, out_hw=(256, 256), order=3):
    n_slices, _, _, n_pcs = data.shape          # 1280,416,416,4
    out = np.empty((n_slices, *out_hw, n_pcs), dtype=np.complex64)
    
    for z in range(n_slices):                   # process one slice at a time
        for pc in range(n_pcs):
            out[z, :, :, pc] = resize_complex(data[z, :, :, pc],
                                              out_hw, order)
    return out

In [ ]:
resized = batch_resize(dataset['M'], out_hw=(256, 256), order=3)

In [19]:
print(resized.shape, resized.dtype)             # (1280, 256, 256, 4) complex64
print(f"Memory size: {resized.nbytes / 1000000000} GB")

(1280, 256, 256, 4) complex64
Memory size: 2.68435456 GB


In [17]:
if 'dataset' in locals():
    del dataset

In [ ]:
outputs = {}
for mode in deepssfp.dataset.modes:
    print(f"\n===== Training: {mode} =====")
    ds = deepssfp.dataset.Dataset(mode, input_data=resized)
    model_dict = deepssfp.analysis.run_training(ds, mode, model_name, model_dir, train_model=True, epochs=800, verbose=False)
    pred, target = deepssfp.analysis.run_inference_on_test_dataset(ds, model_dict['model'], verbose=False)
    metrics = deepssfp.analysis.compute_image_metrics(pred, target, verbose=True)
    outputs[mode] = metrics

In [ ]:
outputs = {}
for mode in deepssfp.dataset.modes:
    print(f"\n===== Results: {mode} =====")
    pass

    model_path = os.path.join(model_dir, f"{model_name}_{mode.lower().replace(':', '_')}")
    history_path = f"{model_path}_history.npz"
    if os.path.exists(history_path):
        history_data = np.load(history_path)
        history_dict = {}
        for key in history_data.files:
            history_dict[key] = history_data[key].tolist()
    else:
        history_dict = None
    
    deepssfp.plot_training_history(
        history_dict
    )